In [ ]:
import datetime
import dijible
import dotenv
dotenv.load_dotenv()

from datetime import timedelta

from plotly import express as plotly_express
import pandas
pandas.options.display.max_rows = 100
pandas.options.display.min_rows = 100
from datetime import tzinfo

In [ ]:
POUNDS_TO_KILOGRAMS = 0.45359237
KILOGRAMS_TO_POUNDS = 1 / POUNDS_TO_KILOGRAMS

HEIGHT = 167.74

WINDOW_SIZE_DAYS = 7
TARGET_DATE = datetime.datetime(2026, 6, 21)
TARGET_WEIGHT = 139 * POUNDS_TO_KILOGRAMS

In [ ]:
intake_df = dijible.get_nutrient_log("calories")

In [ ]:
# df = notion.get_weight_log(timedelta=timedelta(days=7))
df = dijible.get_weight_log(timedelta=timedelta(days=7))
weight = df["weight"].mean() * POUNDS_TO_KILOGRAMS
weight

In [ ]:
intake_df["date"] = (intake_df["consumed_at"].dt.tz_convert("America/Los_Angeles") - datetime.timedelta(hours=5)).dt.date
calories_by_day = intake_df[intake_df["calories"] > 0].groupby("date").sum()
calories_allotted_by_day = intake_df[intake_df["calories"] < 0].groupby("date").sum()

In [ ]:
# Estimate activity factor from actual data

# Calculate current weight loss
# df = notion.get_weight_log()
df = dijible.get_weight_log()
df["date"] = df["date"].dt.tz_convert("America/Los_Angeles").dt.date
df.sort_values("date", inplace=True)
# Calculate 7-day moving average, centered on the current day
df["7-day moving average"] = df["weight"].rolling(window=7, center=True).mean()
# Calculate the moving average weight loss per day by subtracting the beginning and end moving average weight of each window
df["weight loss per day"] = (df["7-day moving average"].shift(-3) - df["7-day moving average"])/3
# Drop all rows with nans in 7-day moving average
moving_average_df = df.dropna(subset=["7-day moving average"])
weight_loss = (moving_average_df["7-day moving average"].values[0] - moving_average_df["7-day moving average"].values[-1])
weight_loss_per_day =  weight_loss / (moving_average_df["date"].values[-1] - moving_average_df["date"].values[0]).days
df["calorie_deficit_based_on_loss"] = df["weight loss per day"] * 3500
# What number of maintenance calories leads to this calorie deficit?
calories_consumed = sum([x for x in intake_df["calories"] if x > 0])
calories_allotted = sum([x for x in intake_df["calories"] if x < 0])
calories_consumed_per_day = calories_consumed / (len(df) - 1) # don't count today
# maintenance_calories = calories_consumed_per_day + calorie_deficit
# maintenance_calories

# Add a column to df containing the calories consumed on that day
df["calories_consumed"] = [calories_by_day.loc[x, "calories"] if x in calories_by_day.index else 0 for x in df["date"]]
df["calories_allotted"] = [calories_allotted_by_day.loc[x, "calories"] if x in calories_allotted_by_day.index else 0 for x in df["date"]]
df["calories_consumed_moving_average"] = df["calories_consumed"][0:-1].rolling(window=7, center=True).mean()
df["maintenance_calories_estimated"] = df["calories_consumed_moving_average"] - df["calorie_deficit_based_on_loss"]
df["exercise_factor"] = df["maintenance_calories_estimated"] / (10 * df["7-day moving average"] *POUNDS_TO_KILOGRAMS + 6.25 * HEIGHT - 5 * 37 + 5) - 1


exercise_factor = df["exercise_factor"].mean()
exercise_factor

In [ ]:
def calculate_maintenance_calories(weight, height, exercise_factor):

    bmr = 10 * weight + 6.25 * height - 5 * 38 + 5

    return bmr * (1 + exercise_factor)

starting_maintenance_calories = calculate_maintenance_calories(weight, HEIGHT, exercise_factor)
ending_maintenance_calories = calculate_maintenance_calories(TARGET_WEIGHT, HEIGHT, exercise_factor)

# Given the starting and ending dates and target, calculate the weight loss
# that needs to happen per day to reach the goal

def calculate_weight_loss_per_day(starting_weight, ending_weight, starting_date, ending_date):
    
        days = (ending_date.date() - starting_date.date()).days
    
        weight_loss = starting_weight - ending_weight
    
        return weight_loss / days

kg_per_day = calculate_weight_loss_per_day(weight, TARGET_WEIGHT, datetime.datetime.now(), TARGET_DATE)
pounds_per_day = kg_per_day * KILOGRAMS_TO_POUNDS
calorie_deficit = pounds_per_day * 3500

print(starting_maintenance_calories - calorie_deficit)

In [ ]:
# Make a plot of weight loss and calories consumed per day with different y axes
fig = plotly_express.line(df, x="date", y=["weight loss per day", "calories_consumed_moving_average"])
# fig.update_yaxes(title_text="Weight loss per day (pounds)")
fig.update_yaxes(title_text="Calories consumed per day")

In [ ]:
# Make a plot of weight loss and calories consumed per day with different y axes
fig = plotly_express.line(df, x="date", y=["7-day moving average"])
# fig.update_yaxes(title_text="Weight loss per day (pounds)")
fig.update_yaxes(title_text="Weight (pounds)")

In [ ]:
# Make a plot of weight loss and calories consumed per day with different y axes
df["exercise_factor_moving_average"] = df["exercise_factor"][0:-1].rolling(window=90, center=True).mean()
fig = plotly_express.line(df, x="date", y=["exercise_factor_moving_average"])
# fig.update_yaxes(title_text="Weight loss per day (pounds)")
fig.update_yaxes(title_text="Exercise factor per day")